# Series of target sweeps: power tuning and responsivity

Many measurements are a series of target sweeps, one per value of something that changes: the tone power (power tuning), the blackbody temperature (responsivity), the bath temperature, ... This notebook shows the tools for taking, loading, and fitting these series without per-notebook loading code.

## Data format

Each series point is one `citkid.crs.procedures.target_sweep`, saved to its own group named `set{i}` (`citkid.zarr_util.set_name(i)`):

```
data.zarr/
    set0/            fres, qres, ares, res_idxs, fine_sweep/{f, z}, gain_sweep/{f, z}, (ts_00/ ...)
    set1/
    ...
```

`citkid.zarr_util.list_sets(root)` returns the indices of the `set{i}` groups in a store, in numerical order, ignoring any other groups.

## Tools

| Function | What it does |
| --- | --- |
| `citkid.crs.procedures.power_sweep` | Takes a target sweep at each tone power, saving `set{i}` groups |
| `sweep_series.open_series_datasets` | Opens one `DataSet` per set, with the loading steps embedded |
| `sweep_series.run_series_analysis` | Runs the analysis path on every set, without the UI |
| `sweep_series.a_vs_power_fit` | `SeriesXYFit` of the nonlinearity parameter a vs tone power |
| `sweep_series.responsivity_fit` | `SeriesXYFit` of fr vs optical power (integrated responsivity) |
| `sweep_series.pick_optimal_ares` | Picks each resonator's optimal tone power |
| `sweep_series.ares_for_a` | Tone power at which the a vs power fit reaches a target a |
| `sweep_series.save_popt` | Saves the optimal powers and sweeps |
| `sweep_series.target_sweep_steps` | The loading steps themselves, for custom uses |

`sweep_series` is `citkid.pipeline.sweep_series`.

# Part 1: Power tuning

## Take the power sweep

`power_sweep` runs `target_sweep` once per power (see `crs_daq.ipynb` for the CRS setup and the `target_sweep` parameters). `aress` can be 1D (every tone gets the same powers) or 2D, shape (n_sets, ntones). With `shuffle = True`, each tone's powers are randomly reordered across the sets; the fitting and picking below handle that.

In [ ]:
import os
import numpy as np
import zarr
from citkid.crs.instrument import CRS
from citkid.crs.procedures import power_sweep

main_dir = '/path/to/data'

crs = CRS(serial_number = 46, interface = 'enp5s0')
await crs.configure_system(clock_source = 'VCXO', full_scale_dbm = 7,
                           analog_bank_high = False)
await crs.set_nco({2: 870e6})

root_prep = zarr.open(os.path.join(main_dir, 'daq_prep.zarr'), mode = 'r')
fres = root_prep['fres_init'][...]
qres = root_prep['qres_init'][...]
res_idxs = root_prep['res_idxs'][...]

root = zarr.open(os.path.join(main_dir, 'popt.zarr'), mode = 'a')
_ = await power_sweep(
    crs, fres, np.arange(-60., -29., 2), qres, res_idxs, root,
    ch_map = {2: np.arange(len(fres))},
    npoints_rough = None,
    npoints_gain = 50,
    npoints_fine = 1000,
    gain_span_factor = 10,
    nsamps = 1000,
)

## Fit the IQ loops

`open_series_datasets(main_dir, data_path, out_root, ...)` opens one `DataSet` per set, stored in `out_root/set{i}`. The loading steps are embedded in each `DataSet`, so later you can reopen them from the zarr path alone, and running the same cell again loads the existing results. `res_idxs` must be the same in every set.

Loading options (all optional):

- `fres_all`, `qres_all`: every resonance in the band, used to cut neighbouring resonances out of the fits. `fres_all` is `'<store>.zarr/<key>'` (store relative to `main_dir`) or `'<key>'` for an array in the root of the data store. `qres_all` is given the same way, or as one number for every resonance. By default the swept tones' own `fres` and `qres` are used.
- `f_avg`: average this many adjacent fine sweep points together.
- `ts`, `ts_nsamps`: timestream group and number of samples, for the `'ts'` calibration (Part 2).
- `set_idxs`: which sets to open, in series order (default: all). `prefix`: group name prefix, e.g. `'amp'` for data saved as `amp{i}`.

Keep `out_root` separate from the raw data store, so the raw data can stay read-only.

`a_vs_power_fit` fits a = a_slope * 10 ** (P / 20) + a_offset (linear in tone amplitude) for each resonator, leaving out every point from the first one with a > `a_max` on. The fits are saved to `out_root['xy_fit/popt']`, shape (nrows, 2).

In [ ]:
from citkid.pipeline.interactive import run_iq_series
from citkid.pipeline.sweep_series import open_series_datasets, a_vs_power_fit

root_out = zarr.open(os.path.join(main_dir, 'popt_fits.zarr'), mode = 'a')
datasets = open_series_datasets(
    main_dir, 'popt.zarr', root_out,
    fres_all = 'daq_prep.zarr/fres_all',
    qres_all = 'daq_prep.zarr/qres_all',
    f_avg = 4,
)

run_iq_series(
    datasets = datasets,
    analysis_yaml_path = 'iq',
    root = root_out,
    x_param_name = 'ares',
    x_name = 'Tone power (dBm)',
    y_param_name = 'a',
    xy_fit = a_vs_power_fit(a_max = 0.8),
)

## Pick the optimal powers

`pick_optimal_ares(datasets, a_max)` orders each resonator's points by power and takes the first point with a >= `a_max` as bifurcated. Of the points below it, it picks the one with the largest a. Points without a fit or marked bad in the window are skipped. It returns the optimal power of each resonator and the index (into `datasets`) of its optimal point.

Resonators with no usable point get NaN. **Fallbacks and limits are up to you**: replace the NaNs and adjust the other powers before saving them.

Alternatively, `ares_for_a(popt, a_target)` gives the power where each resonator's a vs power fit reaches `a_target`, which can lie between the measured powers.

In [ ]:
from citkid.pipeline.sweep_series import (
    open_series_datasets, pick_optimal_ares, ares_for_a, save_popt
)

# The DataSets are reopened with the same options (or reuse datasets from above)
datasets = open_series_datasets(
    main_dir, 'popt.zarr', root_out,
    fres_all = 'daq_prep.zarr/fres_all',
    qres_all = 'daq_prep.zarr/qres_all',
    f_avg = 4,
)
ares_opt, opt_idxs = pick_optimal_ares(datasets, a_max = 0.5)

# Or, from the fits:
# ares_opt = ares_for_a(root_out['xy_fit/popt'][...], a_target = 0.5)

# Your own fallback and limits
ares_opt[np.isnan(ares_opt)] = -46.
ares_opt = np.clip(ares_opt, -50., -40.)

# Saves res_idxs, ares_opt, opt_idxs, and the fine sweeps at the optimal points
save_popt(root_out.require_group('popt'), datasets, ares_opt, opt_idxs)

In [ ]:
# Send the optimal powers to daq_prep
root_prep = zarr.open(os.path.join(main_dir, 'daq_prep.zarr'), mode = 'a')
root_prep.create_array('ares', data = ares_opt, overwrite = True)

# Part 2: Responsivity

## Take the blackbody sweep

The blackbody temperature changes outside the CRS, so write the loop yourself, saving each series point to its own `set{i}` group with a target sweep and a timestream. For example:

```python
from citkid.zarr_util import set_name

root = zarr.open(os.path.join(main_dir, 'bb_sweep_00.zarr'), mode = 'a')
ch_map = None
for set_idx, T in enumerate(BBTs):
    ...  # set and wait for the blackbody temperature
    grp = root.require_group(set_name(set_idx))
    fres, ch_map = await target_sweep(crs, fres, ares, qres, res_idxs, grp,
                                      ch_map = ch_map, npoints_fine = 1000)
    await crs.capture_ts(fres, ares, ts_duration_s, dec_stage,
                         grp.require_group('ts_00'), ch_map = ch_map)
```

## Calibrate and analyze every set

With `ts = 'ts_00'`, the timestream is loaded too, for the `'ts'` calibration. `run_series_analysis` runs the full analysis path on every set without the UI, e.g. on the on-resonance tones only.

In [ ]:
from citkid.pipeline.sweep_series import open_series_datasets, run_series_analysis

root_out = zarr.open(os.path.join(main_dir, 'bb_sweep_00_analysis.zarr'), mode = 'a')
datasets = open_series_datasets(
    main_dir, 'bb_sweep_00.zarr', root_out,
    cal_yaml_path = 'ts',
    fres_all = 's21_full.zarr/fres_all',
    qres_all = 6000,
    ts = 'ts_00',
    ts_nsamps = 10_000,
)

on_res = np.where(datasets[0].res_idxs[:] >= 0)[0]
_ = run_series_analysis(datasets, analysis_yaml_path = 'iq', data_idxs = on_res)

## Fit the responsivity

The blackbody power isn't stored in the DataSets, so pass it with `x_values`, one value per set (in W). `responsivity_fit` fits fr = f1 * (1 + x(P; R0, P0, x0)) (`citkid.responsivity.funcs.responsivity_int_x0`), where f1 is fr at the lowest power. It uses guesses, which you can edit in the window. The fits are saved to `root_out['xy_fit']`: `popt` and `p0`, shape (nrows, 4), in the order [resp_R0, resp_P0, resp_x0, resp_f1].

In [ ]:
from citkid.pipeline.interactive import run_iq_series
from citkid.pipeline.sweep_series import responsivity_fit

BBT = ...   # blackbody temperature of each set
BBP = ...   # blackbody power of each set (W), e.g. interpolated from a calibration

run_iq_series(
    datasets = datasets,
    analysis_yaml_path = 'iq',
    root = root_out,
    x_values = BBP,
    x_name = 'BBP (W)',
    y_param_name = 'fr',
    data_idxs = on_res,
    xy_fit = responsivity_fit(),
    xscale = 'log',
)

# Reopening and moving data

- Each `DataSet` embeds a few lines of source that call `target_sweep_steps` with your options. Reopen it with `DataSet(zarr_path = root_out['set3'])`, or by calling `open_series_datasets` again with the same options. Different options raise an error instead of mixing results.
- If the raw data moves, pass the new `main_dir` to `open_series_datasets`, or `custom_main_dir_overwrite = new_main_dir` to `DataSet`. Keep the data paths relative to `main_dir` for this to work.

# Using the loading steps directly

For other uses, `target_sweep_steps` returns the loading steps as a list of `plStep`s, and `target_sweep_source` returns them as source code for `DataSet(custom_source = ...)`:

```python
from citkid.pipeline import DataSet
from citkid.pipeline.sweep_series import target_sweep_steps, target_sweep_source

# e.g. in make_custom_steps for run_iq_series
def make_custom_steps(series_idx):
    return target_sweep_steps('popt.zarr', f'set{series_idx}', main_dir = main_dir, f_avg = 4)

# a single DataSet, reopenable without a custom steps file
DS = DataSet(zarr_path = 'single.zarr', cal_yaml_path = 'iq',
             custom_source = target_sweep_source(main_dir, 'popt.zarr', 'set0'))
```